## Create 5 different way to split the data between clients, test and val sets

in folder split_i there is a file with the list of test case_ids, a file with the list of the val case_ids, and 4 folders, one for each client created. In each client the train set. No other data from the datasets is saved in splits

(old) in folder split_i there is a file with the list of test case_ids, and 4 folders, one for each client created. In each client there is the division between train and val set: a single file with the list of case_ids for each set. No other data from the datasets is saved in splits

In [14]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from random import shuffle
from functools import reduce

### get utils

In [15]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split

from src.zoorvival.data import load_tcga_data
from src.zoorvival.nn.training import as_torch_dataset

%reload_ext autoreload
%autoreload 2

np.random.seed(42)

In [16]:
PROJECT = "BLCA"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [17]:
def request_tcga_file_info(data_type: str, project_ids=None) -> pd.DataFrame:
    """
    Get TCGA file information from GDC API.

    Args:
        data_type (str): The type of data to retrieve.
        project_ids (list[str], optional): List of project IDs to filter by. Defaults to None.
    
    Returns:
        pd.DataFrame: DataFrame containing file information.
    """
    if project_ids is None:
        project_ids = get_tcga_projects()
    fields = [
        "file_name",
        "md5sum",
        "cases.submitter_id",
        "cases.samples.sample_type",
        "cases.project.project_id",
        "cases.project.primary_site",
    ]
    fields = ",".join(fields)
    files_endpt = "https://api.gdc.cancer.gov/files"
    filters = {
        "op": "and",
        "content": [
            {
                "op": "in",
                "content": {
                    "field": "files.experimental_strategy",
                    "value": [data_type],
                },
            },
            {
                "op": "in",
                "content": {
                    "field": "cases.project.project_id",
                    "value": project_ids,
                },
            },
        ],
    }
    params = {"filters": filters, "fields": fields, "format": "TSV", "size": "200000"}
    response = requests.post(
        files_endpt, headers={"Content-Type": "application/json"}, json=params
    )
    return pd.read_csv(StringIO(response.content.decode("utf-8")), sep="\t")


def process_tcga_file_info(df_files, suffix, data_type) -> pd.DataFrame:
    rename_cols = {
        "cases.0.project.project_id": "project_id",
        "cases.0.project.primary_site": "primary_site",
        "cases.0.submitter_id": "submitter_id",
        "cases.0.samples.0.sample_type": "sample_type",
        "file_name": f"{data_type}_file_name",
        "id": f"{data_type}_file_id",
        "md5sum": f"{data_type}_md5sum",
    }
    df_files = df_files.rename(columns=rename_cols)
    df_files = df_files[df_files[f"{data_type}_file_name"].str.endswith(suffix)]
    df_files = df_files[df_files["sample_type"] == "Primary Tumor"]
    df_files = df_files[~df_files.duplicated(subset=["submitter_id"], keep="first")]
    return df_files


def get_suffix_counts(df_files) -> pd.Series:
    file_col = [c for c in df_files.columns if "file_name" in c][0]
    file_suffixes = [".".join(f.split(".")[-2:]) for f in df_files[file_col]]
    suffix_counts = pd.Series(file_suffixes).value_counts()
    suffix_counts = suffix_counts[suffix_counts > 1]
    suffix_counts = suffix_counts.reset_index()
    return suffix_counts

def get_tcga_projects() -> list[str]:
    """
    Get a list of TCGA projects from the GDC API.
    """
    response = requests.get("https://api.gdc.cancer.gov/projects", params={"size": 10000})
    response.raise_for_status()
    projects = response.json()["data"]["hits"]
    projects = [proj for proj in projects if proj.get("project_id", "").startswith("TCGA-")]
    projects = [proj["id"] for proj in projects]
    return projects

### start

In [18]:
# get the whole data
study = 'BLCA'

if study in ['LUAD', 'UCEC', 'BRCA', 'BLCA']:
    omics_data_c = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/other/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )
    omics_data_h = omics_data_c
    omics_data_x = omics_data_c

else:
    # omics data
    omics_data_c = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/combine/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )

    omics_data_h = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/hallmarks/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )

    omics_data_x = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/xena/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )

# metadata
label_data = pd.read_csv(f'src/datasets_csv/metadata/tcga_{study}.csv', low_memory=False)    # contains pairing between case_id and slide_id(s) + info on survival

# clinical data
clinical_data = pd.read_csv(f'src/datasets_csv/clinical_data/tcga_{study}_clinical.csv', index_col=0) # clinical data


In [19]:
db = load_tcga_data(PROJECT)

In [20]:
train_patients = list('TCGA-' + i for i in db.train.df_clinical.index.astype(str))
test_patients = list('TCGA-' + i for i in db.test.df_clinical.index.astype(str))

train_patients.extend(test_patients)
wsi_patients = train_patients
wsi_patients = list(set(wsi_patients))
print('number of wsi present in wsi database: ', len(wsi_patients))

number of wsi present in wsi database:  369


In [21]:
def save_test_set(test_ids, destination, what):
    print(f'size of {what} ids: ', test_ids.shape)
    test_df = pd.DataFrame(test_ids, columns=[what])
    # save test set
    test_df.to_csv(f'{destination}')

In [22]:
def get_stratifier(all_ids, common_ids, clinical_only_ids, omics_c_only_ids, label_data):
    y = []
    #events = int(events)
    censorship = 'censorship_dss'
    if study in ['LUAD', 'UCEC', 'BRCA', 'BLCA']:
        censorship = 'censorship_os'
    for i, id in enumerate(all_ids):
        if id in common_ids:
            censored = (label_data.loc[label_data['case_id'] == id, censorship] == 1).any()
            has_wsi = True if id in wsi_patients else False
            if censored and has_wsi:
                y.insert(i, 0)
            elif censored and not has_wsi:
                y.insert(i, 1)
            elif not censored and has_wsi:
                y.insert(i, 2)
            else:
                y.insert(i, 3)
        elif clinical_only_ids is not None and id in clinical_only_ids and len(clinical_only_ids)> 1:
            y.insert(i, 4)
        elif omics_c_only_ids is not None and id in omics_c_only_ids and len(omics_c_only_ids)>1:
            y.insert(i, 4)
        elif omics_c_only_ids is not None and id in omics_c_only_ids and len(omics_c_only_ids)<= 1:
            y.insert(i, 2)
        else:
            print('something is wrong')
            y.insert(i, None)
    
    return y

In [23]:
def count_wsi(list_ids):
    num_wsi = 0
    num_wsi_missing = 0
    for i in list_ids:
        if i in wsi_patients:
            num_wsi += 1
        else:
            num_wsi_missing +=1

    print('number patients with wsi: ', num_wsi)
    print('number patients without wsi: ', num_wsi_missing)

In [24]:
# create all folders
num_splits = 5
num_clients = 3
random_state = 42
general_dir = f'src/splits_{study}'
os.makedirs(general_dir, exist_ok=True)

for split in range(num_splits):

    split_dir = general_dir + '/split_' + str(split)
    os.makedirs(split_dir, exist_ok=True)

    # start with test set - 0.15 of whole data
    # get whole data available
    common_ids = label_data["case_id"].unique()
    #np.random.shuffle(common_ids)
    print(f"Total number of patients in label_data: {len(common_ids)}")

    # add ids of other sets too, the not common ones
    omics_c_only_ids = [pid for pid in omics_data_c.index if pid not in common_ids]
    print(f"Number of patient_ids in omics data combine but not in label data: {len(omics_c_only_ids)}")

    clinical_only_ids = [pid for pid in clinical_data['case_id'] if pid not in common_ids and pid not in omics_c_only_ids]
    print(f"Number of patient_ids in clinical data but not in label data and not in omics: {len(clinical_only_ids)}")

    all_ids = reduce(np.union1d, (common_ids, clinical_only_ids, omics_c_only_ids))
    
    # make sure that the test set is balanced between the 3 sets and the events and divide
    y = get_stratifier(all_ids, common_ids, clinical_only_ids, omics_c_only_ids, label_data)
    all_ids, test_ids, y_train, y_test  = train_test_split(all_ids, y, test_size=0.15, random_state=random_state, stratify=y)

    print('num wsi test set:')
    count_wsi(test_ids)
    
    # save the test set
    test_dir = split_dir + '/test.csv'
    save_test_set(test_ids, test_dir, 'test')
    print('data for training and val size: ', all_ids.size)

    print('num wsi train-val set:')
    count_wsi(all_ids)

    # get validation data, global
    y = get_stratifier(all_ids, common_ids, clinical_only_ids, omics_c_only_ids, label_data)
    all_ids, val_ids, y_train, y_test  = train_test_split(all_ids, y, test_size=0.15, random_state=random_state, stratify=y)

    # save the val set
    val_dir = split_dir + '/val.csv'
    save_test_set(val_ids, val_dir, 'val')
    print('data for val set size: ', val_ids.size)
    print('data for train set size: ', all_ids.size)

    print('num wsi val set:')
    count_wsi(val_ids)

    print('num wsi all train set:')
    count_wsi(all_ids)

    # get train/val data for the clients divided by subset
    clinical_only_ids_clients = [pid for pid in clinical_only_ids if pid in all_ids]
    common_ids_clients = [pid for pid in common_ids if pid in all_ids]
    omics_c_only_ids_clients = [pid for pid in omics_c_only_ids if pid in all_ids]

    # shuffle
    shuffle(clinical_only_ids_clients)
    shuffle(common_ids_clients)
    shuffle(omics_c_only_ids_clients)

    # get split for this ids
    clinical_only_splits = np.array_split(clinical_only_ids_clients, num_clients)
    print('division of data in clients, clinical only data: ', [len(split) for split in clinical_only_splits])
    common_splits = np.array_split(common_ids_clients, num_clients)
    print('division of data in clients, common data: ', [len(split) for split in common_splits])
    omics_c_only_splits = np.array_split(omics_c_only_ids_clients, num_clients)
    print('division of data in clients, omics only data: ', [len(split) for split in omics_c_only_splits])
    print('test: ', [split[:3] for split in omics_c_only_splits])

    # create the clients
    for i in range(num_clients):
        client_folder = f'{split_dir}/client_{i}'
        os.makedirs(client_folder, exist_ok=True)

        client_data = reduce(np.union1d, (clinical_only_splits[i], common_splits[i], omics_c_only_splits[i]))

        print('wsi for client ', i)
        count_wsi(client_data)

        # save train set
        train_dir = client_folder + '/train.csv'
        save_test_set(client_data, train_dir, 'train')
    
    random_state += 1


Total number of patients in label_data: 371
Number of patient_ids in omics data combine but not in label data: 2
Number of patient_ids in clinical data but not in label data and not in omics: 0
num wsi test set:
number patients with wsi:  56
number patients without wsi:  0
size of test ids:  (56,)
data for training and val size:  317
num wsi train-val set:
number patients with wsi:  313
number patients without wsi:  4
size of val ids:  (48,)
data for val set size:  48
data for train set size:  269
num wsi val set:
number patients with wsi:  48
number patients without wsi:  0
num wsi all train set:
number patients with wsi:  265
number patients without wsi:  4
division of data in clients, clinical only data:  [0, 0, 0]
division of data in clients, common data:  [89, 89, 89]
division of data in clients, omics only data:  [1, 1, 0]
test:  [array(['TCGA-GD-A76B'], dtype='<U12'), array(['TCGA-GV-A3QG'], dtype='<U12'), array([], dtype='<U12')]
wsi for client  0
number patients with wsi:  88


# now consider the centralized dataset

In [25]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from random import shuffle
from functools import reduce

In [26]:
num_splits = 5
num_clients = 3

for split in range(num_splits):
    split_dir = f'src/splits_{study}/split_{split}'

    # create new folder
    cent_folder = f'{split_dir}/client_0_cent'
    os.makedirs(cent_folder, exist_ok=True)

    # create new dataFrame to save case_id
    train_c = pd.DataFrame(columns = ['train'])
    print(train_c)

    # get the 
    for i in range(num_clients):
        client_folder = f'{split_dir}/client_{i}'
        df = pd.read_csv(f'{client_folder}/train.csv', index_col=0)
    
        train_c = pd.concat([train_c, df], ignore_index=True)
        train_c = train_c.apply(
            lambda col: pd.Series(col.dropna().tolist() + [None] * col.isna().sum())
        )
        print(f'append from client {i}: \n {train_c}')
    
    train_c.to_csv(f'{cent_folder}/train.csv')


    
        


Empty DataFrame
Columns: [train]
Index: []
append from client 0: 
            train
0   TCGA-2F-A9KP
1   TCGA-4Z-AA7W
2   TCGA-BL-A0C8
3   TCGA-BL-A13I
4   TCGA-BT-A20N
..           ...
85  TCGA-ZF-A9RF
86  TCGA-ZF-A9RN
87  TCGA-ZF-AA4T
88  TCGA-ZF-AA54
89  TCGA-ZF-AA5H

[90 rows x 1 columns]
append from client 1: 
             train
0    TCGA-2F-A9KP
1    TCGA-4Z-AA7W
2    TCGA-BL-A0C8
3    TCGA-BL-A13I
4    TCGA-BT-A20N
..            ...
175  TCGA-XF-AAN2
176  TCGA-XF-AAN3
177  TCGA-XF-AAN5
178  TCGA-YC-A8S6
179  TCGA-ZF-AA52

[180 rows x 1 columns]
append from client 2: 
             train
0    TCGA-2F-A9KP
1    TCGA-4Z-AA7W
2    TCGA-BL-A0C8
3    TCGA-BL-A13I
4    TCGA-BT-A20N
..            ...
264  TCGA-ZF-AA4R
265  TCGA-ZF-AA4U
266  TCGA-ZF-AA4W
267  TCGA-ZF-AA51
268  TCGA-ZF-AA53

[269 rows x 1 columns]
Empty DataFrame
Columns: [train]
Index: []
append from client 0: 
            train
0   TCGA-2F-A9KO
1   TCGA-2F-A9KP
2   TCGA-4Z-AA7M
3   TCGA-4Z-AA7N
4   TCGA-4Z-AA7S
..       